In [1]:
import pandas as pd
import numpy as np

tournament = pd.read_csv('Tournament Matchups.csv')

tournament.head()

,YEAR,BY YEAR NO,TEAM NO,TEAM,SEED,ROUND,CURRENT ROUND,SCORE
0,2025,2140,1141,Auburn,1,4,64,83
1,2025,2139,1145,Alabama St.,16,64,64,63
2,2025,2138,1119,Louisville,8,64,64,75
3,2025,2137,1134,Creighton,9,32,64,89
4,2025,2136,1114,Michigan,5,16,64,68


In [2]:
# Build game-level dataset: every two rows are one game within each YEAR
# Team A = first row, Team B = second row
# Keep years in descending order so recent tournaments appear first
_t = tournament.sort_values(['YEAR', 'BY YEAR NO'], ascending=[False, False]).reset_index(drop=True)

# Create game pairing keys inside each year
_t['_pair_idx'] = _t.groupby('YEAR').cumcount() // 2
_t['_side'] = np.where(_t.groupby('YEAR').cumcount() % 2 == 0, 'A', 'B')

# Split to A/B then merge into one row per game
A = _t[_t['_side'] == 'A'].copy()
B = _t[_t['_side'] == 'B'].copy()

games = A.merge(
    B,
    on=['YEAR', '_pair_idx'],
    how='inner',
    suffixes=('_A', '_B'),
    validate='one_to_one'
)

# Final schema requested
tournament_games = games.rename(columns={
    'TEAM NO_A': 'ATeamNo',
    'TEAM_A': 'ATeam',
    'TEAM NO_B': 'BTeamNo',
    'TEAM_B': 'BTeam',
    'SEED_A': 'ASeed',
    'SEED_B': 'BSeed',
    'ROUND_A': 'ARound',
    'ROUND_B': 'BRound',
    'SCORE_A': 'AScore',
    'SCORE_B': 'BScore'
})[
    ['YEAR', 'ATeamNo', 'ATeam', 'BTeamNo', 'BTeam', 'ASeed', 'BSeed', 'ARound', 'BRound', 'AScore', 'BScore']
].copy()

# Pre-game matchup feature: positive means Team B has a worse seed number
# (e.g., 16 - 1 = 15 favors Team A)
tournament_games['SeedDiff'] = tournament_games['BSeed'] - tournament_games['ASeed']

tournament_games['ATeamWin'] = (tournament_games['AScore'] > tournament_games['BScore']).astype(int)

tournament_games.head(10)

,YEAR,ATeamNo,ATeam,BTeamNo,BTeam,ASeed,BSeed,ARound,BRound,AScore,BScore,SeedDiff,ATeamWin
0,2025,1141,Auburn,1145,Alabama St.,1,16,4,64,83,63,15,1
1,2025,1119,Louisville,1134,Creighton,8,9,64,32,75,89,1,0
2,2025,1114,Michigan,1089,UC San Diego,5,12,16,64,68,65,7,1
3,2025,1092,Texas A&M,1080,Yale,4,13,32,64,80,71,9,1
4,2025,1112,Mississippi,1104,North Carolina,6,11,16,64,71,64,5,1
5,2025,1124,Iowa St.,1120,Lipscomb,3,14,32,64,82,55,11,1
6,2025,1118,Marquette,1106,New Mexico,7,10,64,32,66,75,3,0
7,2025,1113,Michigan St.,1139,Bryant,2,15,8,64,87,62,13,1
8,2025,1131,Florida,1105,Norfolk St.,1,16,1,64,95,69,15,1
9,2025,1135,Connecticut,1103,Oklahoma,8,9,32,64,67,59,1,1


In [3]:
import warnings

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, log_loss

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB

# -----------------------------
# Features / target
# -----------------------------
y = tournament_games['ATeamWin'].astype(int)

feature_cols = ['ASeed', 'BSeed', 'SeedDiff', 'ATeamNo', 'BTeamNo']
if 'PowerRatingDiff' in tournament_games.columns:
    feature_cols.append('PowerRatingDiff')

X = tournament_games[feature_cols].copy()

# Treat team IDs as categorical labels
X['ATeamNo'] = X['ATeamNo'].astype(str)
X['BTeamNo'] = X['BTeamNo'].astype(str)

numeric_cols = [c for c in ['ASeed', 'BSeed', 'SeedDiff', 'PowerRatingDiff'] if c in X.columns]
categorical_cols = ['ATeamNo', 'BTeamNo']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -----------------------------
# Preprocessors
# -----------------------------
preprocess_scaled = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_cols),
    ]
)

preprocess_tree = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
        ]), numeric_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_cols),
    ]
)

preprocess_nb = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scale', MinMaxScaler()),
        ]), numeric_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_cols),
    ]
)

# -----------------------------
# Model pipelines
# -----------------------------
models = {
    'Logistic Regression': Pipeline(steps=[
        ('preprocess', preprocess_scaled),
        ('model', LogisticRegression(max_iter=1000, solver='lbfgs')),
    ]),
    'Neural Network': Pipeline(steps=[
        ('preprocess', preprocess_scaled),
        ('model', MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation='relu',
            solver='adam',
            alpha=1e-4,
            max_iter=500,
            random_state=42,
            early_stopping=True,
            n_iter_no_change=10,
        )),
    ]),
    'SVM': Pipeline(steps=[
        ('preprocess', preprocess_scaled),
        ('model', SVC(kernel='rbf', C=1.0, probability=True, random_state=42)),
    ]),
    'Random Forest': Pipeline(steps=[
        ('preprocess', preprocess_tree),
        ('model', RandomForestClassifier(
            n_estimators=500,
            random_state=42,
            class_weight='balanced_subsample',
            n_jobs=-1,
        )),
    ]),
    'Naive Bayes': Pipeline(steps=[
        ('preprocess', preprocess_nb),
        ('model', MultinomialNB(alpha=1.0)),
    ]),
}

# Suppress noisy sklearn runtime warnings from internal matrix ops
warnings.filterwarnings('ignore', category=RuntimeWarning, module='sklearn.utils.extmath')
warnings.filterwarnings('ignore', category=RuntimeWarning, module='sklearn.linear_model._linear_loss')

# -----------------------------
# Fit/evaluate all models
# -----------------------------
rows = []
for model_name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)

    rows.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Log Loss': log_loss(y_test, y_proba, labels=[0, 1]),
    })

results_table = pd.DataFrame(rows).sort_values('Log Loss').reset_index(drop=True)
results_table['Accuracy'] = results_table['Accuracy'].round(4)
results_table['Log Loss'] = results_table['Log Loss'].round(4)

# order results by log loss
results_table = results_table.sort_values("Log Loss")
results_table

,Model,Accuracy,Log Loss
0,Logistic Regression,0.6495,0.6101
1,Neural Network,0.6449,0.6111
2,SVM,0.6729,0.6303
3,Random Forest,0.6682,0.6835
4,Naive Bayes,0.6355,0.7084


In [4]:
# Add KenPom/Barttorvik features using robust ID-based joins
kp = pd.read_csv('KenPom Barttorvik.csv')[
    ['YEAR', 'TEAM NO', 'KADJ EM', 'KADJ O', 'KADJ D', 'BADJ EM', 'BADJ O', 'BADJ D', 'BARTHAG']
].copy()

kp = kp.rename(columns={'TEAM NO': 'TeamNo'})

# Guardrail: each season/team should map to one row only
if kp.duplicated(['YEAR', 'TeamNo']).any():
    raise ValueError('Duplicate YEAR/TEAM NO keys found in KenPom Barttorvik.csv')

# Align key dtypes
tournament_games['ATeamNo'] = tournament_games['ATeamNo'].astype('Int64')
tournament_games['BTeamNo'] = tournament_games['BTeamNo'].astype('Int64')
kp['TeamNo'] = kp['TeamNo'].astype('Int64')

# Merge Team A metrics
tournament_games = tournament_games.merge(
    kp.rename(columns={
        'TeamNo': 'ATeamNo',
        'KADJ EM': 'AKAdjEM',
        'KADJ O': 'AKAdjO',
        'KADJ D': 'AKAdjD',
        'BADJ EM': 'ABAdjEM',
        'BADJ O': 'ABAdjO',
        'BADJ D': 'ABAdjD',
        'BARTHAG': 'ABarthag',
    }),
    on=['YEAR', 'ATeamNo'],
    how='left',
    validate='many_to_one'
)

# Merge Team B metrics
tournament_games = tournament_games.merge(
    kp.rename(columns={
        'TeamNo': 'BTeamNo',
        'KADJ EM': 'BKAdjEM',
        'KADJ O': 'BKAdjO',
        'KADJ D': 'BKAdjD',
        'BADJ EM': 'BBAdjEM',
        'BADJ O': 'BBAdjO',
        'BADJ D': 'BBAdjD',
        'BARTHAG': 'BBarthag',
    }),
    on=['YEAR', 'BTeamNo'],
    how='left',
    validate='many_to_one'
)

# Simple matchup differential features (Team A minus Team B)
tournament_games['KAdjEMDiff'] = tournament_games['AKAdjEM'] - tournament_games['BKAdjEM']
tournament_games['KAdjODiff'] = tournament_games['AKAdjO'] - tournament_games['BKAdjO']
tournament_games['KAdjDDiff'] = tournament_games['AKAdjD'] - tournament_games['BKAdjD']
tournament_games['BAdjEMDiff'] = tournament_games['ABAdjEM'] - tournament_games['BBAdjEM']
tournament_games['BAdjODiff'] = tournament_games['ABAdjO'] - tournament_games['BBAdjO']
tournament_games['BAdjDDiff'] = tournament_games['ABAdjD'] - tournament_games['BBAdjD']
tournament_games['BarthagDiff'] = tournament_games['ABarthag'] - tournament_games['BBarthag']

# Keep only differential KenPom/Barttorvik features
kp_raw_cols = [
    'AKAdjEM', 'AKAdjO', 'AKAdjD', 'ABAdjEM', 'ABAdjO', 'ABAdjD', 'ABarthag',
    'BKAdjEM', 'BKAdjO', 'BKAdjD', 'BBAdjEM', 'BBAdjO', 'BBAdjD', 'BBarthag'
]
tournament_games = tournament_games.drop(columns=kp_raw_cols)

# Show updated dataset
tournament_games.head(10)


,YEAR,ATeamNo,ATeam,BTeamNo,BTeam,ASeed,BSeed,ARound,BRound,AScore,BScore,SeedDiff,ATeamWin,KAdjEMDiff,KAdjODiff,KAdjDDiff,BAdjEMDiff,BAdjODiff,BAdjDDiff,BarthagDiff
0,2025,1141,Auburn,1145,Alabama St.,1,16,4,64,83,63,15,1,44.27350,27.051,-17.2227,44.487,28.528,-15.959,0.706
1,2025,1119,Louisville,1134,Creighton,8,9,64,32,75,89,1,0,3.94400,1.209,-2.7352,4.590,1.008,-3.582,0.044
2,2025,1114,Michigan,1089,UC San Diego,5,12,16,64,68,65,7,1,3.04330,1.497,-1.5462,4.211,1.639,-2.572,0.048
3,2025,1092,Texas A&M,1080,Yale,4,13,32,64,80,71,9,1,13.13720,1.837,-11.3005,12.364,1.769,-10.595,0.178
4,2025,1112,Mississippi,1104,North Carolina,6,11,16,64,71,64,5,1,2.53590,-1.016,-3.5525,1.425,-1.438,-2.863,0.019
5,2025,1124,Iowa St.,1120,Lipscomb,3,14,32,64,82,55,11,1,17.51438,6.772,-10.7426,19.852,8.158,-11.694,0.276
6,2025,1118,Marquette,1106,New Mexico,7,10,64,32,66,75,3,0,4.25990,5.150,0.8906,3.304,4.920,1.616,0.031
7,2025,1113,Michigan St.,1139,Bryant,2,15,8,64,87,62,13,1,26.38884,11.331,-15.0570,24.584,11.080,-13.504,0.430
8,2025,1131,Florida,1105,Norfolk St.,1,16,1,64,95,69,15,1,37.34698,20.737,-16.6104,36.507,21.484,-15.023,0.546
9,2025,1135,Connecticut,1103,Oklahoma,8,9,32,64,67,59,1,1,0.98680,2.903,1.9160,2.767,3.922,1.155,0.028


In [6]:
import warnings

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, log_loss

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB

# -----------------------------
# Features / target
# -----------------------------
y = tournament_games['ATeamWin'].astype(int)

# Base + engineered matchup-difference features
feature_cols = [
    'ASeed', 'BSeed', 'SeedDiff',
    'ATeamNo', 'BTeamNo',
    'KAdjEMDiff', 'KAdjODiff', 'KAdjDDiff',
    'BAdjEMDiff', 'BAdjODiff', 'BAdjDDiff',
    'BarthagDiff'
]

# Include optional diff features if already created in prior cells
for optional_col in ['PowerRatingDiff', 'APRankDiff']:
    if optional_col in tournament_games.columns:
        feature_cols.append(optional_col)

X = tournament_games[feature_cols].copy()

# Treat team IDs as categorical labels
X['ATeamNo'] = X['ATeamNo'].astype(str)
X['BTeamNo'] = X['BTeamNo'].astype(str)

numeric_cols = [c for c in feature_cols if c not in ['ATeamNo', 'BTeamNo']]
categorical_cols = ['ATeamNo', 'BTeamNo']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -----------------------------
# Preprocessors
# -----------------------------
preprocess_scaled = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_cols),
    ]
)

preprocess_tree = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
        ]), numeric_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_cols),
    ]
)

preprocess_nb = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scale', MinMaxScaler()),
        ]), numeric_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_cols),
    ]
)

# -----------------------------
# Model pipelines
# -----------------------------
models = {
    'Logistic Regression': Pipeline(steps=[
        ('preprocess', preprocess_scaled),
        ('model', LogisticRegression(max_iter=1000, solver='lbfgs')),
    ]),
    'Neural Network': Pipeline(steps=[
        ('preprocess', preprocess_scaled),
        ('model', MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation='relu',
            solver='adam',
            alpha=1e-4,
            max_iter=500,
            random_state=42,
            early_stopping=True,
            n_iter_no_change=10,
        )),
    ]),
    'SVM': Pipeline(steps=[
        ('preprocess', preprocess_scaled),
        ('model', SVC(kernel='rbf', C=1.0, probability=True, random_state=42)),
    ]),
    'Random Forest': Pipeline(steps=[
        ('preprocess', preprocess_tree),
        ('model', RandomForestClassifier(
            n_estimators=500,
            random_state=42,
            class_weight='balanced_subsample',
            n_jobs=-1,
        )),
    ]),
    'Naive Bayes': Pipeline(steps=[
        ('preprocess', preprocess_nb),
        ('model', MultinomialNB(alpha=1.0)),
    ]),
}

warnings.filterwarnings('ignore', category=RuntimeWarning, module='sklearn.utils.extmath')
warnings.filterwarnings('ignore', category=RuntimeWarning, module='sklearn.linear_model._linear_loss')

# -----------------------------
# Fit/evaluate all models
# -----------------------------
rows = []
for model_name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)

    rows.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Log Loss': log_loss(y_test, y_proba, labels=[0, 1]),
    })

results_table_v2 = pd.DataFrame(rows).sort_values('Log Loss').reset_index(drop=True)
results_table_v2['Accuracy'] = results_table_v2['Accuracy'].round(4)
results_table_v2['Log Loss'] = results_table_v2['Log Loss'].round(4)

results_table_v2 = results_table_v2.sort_values("Log Loss")
results_table_v2

,Model,Accuracy,Log Loss
0,Logistic Regression,0.6822,0.5798
1,SVM,0.6963,0.6140
2,Random Forest,0.6776,0.6263
3,Neural Network,0.6776,0.6306
4,Naive Bayes,0.6355,0.7365
